# 자동화된 부동산 기망·사기 및 불법 행위 탐지 시스템

이 노트북은 제출/발표용 설명 자료입니다. 판례 기반 데이터, 공개자료 기반 위험 지표, 파생 계약 예시를 이용해 `sklearn.ensemble.BaggingClassifier`를 학습하고, 사용자가 입력한 부동산 계약 정보를 `안전 / 주의 / 위험`으로 점수화하는 과정을 코드와 함께 보여줍니다.

> 주의: 이 프로젝트는 법적 판단을 대신하지 않습니다. 모델 출력은 계약 전 2차 확인을 위한 위험 신호 탐지 결과입니다.

## 1. 프로젝트 구조

- `데이터/real_estate_fraud_cases_filtered.csv`: 판례 중심 원본 데이터
- `데이터/public_risk_indicators.csv`: 공식 공개자료에서 뽑은 위험 지표
- `데이터/derived_contract_cases.csv`: 학습용 파생 계약 데이터
- `models/bagging_risk_model.joblib`: 학습된 Bagging 모델
- `학습과정/metrics.json`: validation/holdout 평가 지표
- `src/risk_detector/risk/scorer.py`: 웹/API에서 사용하는 최종 위험도 점수화 로직

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

# 노트북을 프로젝트 루트 또는 학습과정 폴더 어디에서 실행해도 동작하게 경로를 맞춘다.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "학습과정":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "데이터"
LEARNING_DIR = PROJECT_ROOT / "학습과정"
MODELS_DIR = PROJECT_ROOT / "models"

print(PROJECT_ROOT)

## 2. 학습 데이터 구성

이 프로젝트의 실제 피해자 원천 계약 데이터는 공개되어 있지 않습니다. 그래서 다음 데이터를 결합했습니다.

1. 판례 텍스트와 법률 카테고리
2. HUG, 국토교통부, 등기소, 법령 기준에서 추출한 위험 신호
3. 정상 계약, 경계 계약, 고위험 계약의 파생 예시

라벨은 실제 법원 판결의 사기 인정 여부가 아니라, 계약 전 위험도 탐지 학습을 위한 `0=안전`, `1=주의`, `2=위험` 등급입니다.

In [ ]:
df = pd.read_csv(DATA_DIR / "derived_contract_cases.csv", encoding="utf-8-sig")
quality = json.loads((DATA_DIR / "data_quality_report.json").read_text(encoding="utf-8"))

print("rows:", len(df))
print("columns:", len(df.columns))
print("duplicates:", quality["duplicate_rows"])
df.head(3)

In [ ]:
source_counts = df["source"].value_counts().rename_axis("source").reset_index(name="rows")
label_counts = df["risk_label"].value_counts().sort_index().rename_axis("risk_label").reset_index(name="rows")
display(source_counts)
display(label_counts)

## 3. 공개자료 기반 위험 지표

`public_risk_indicators.csv`는 실제 앱 런타임에서 외부 API를 호출하지 않더라도, 어떤 공식 기준을 모델 피처와 규칙에 반영했는지 남기기 위한 오프라인 근거 자료입니다.

In [ ]:
public_indicators = pd.read_csv(DATA_DIR / "public_risk_indicators.csv", encoding="utf-8-sig")
external_refs = pd.read_csv(DATA_DIR / "external_case_references.csv", encoding="utf-8-sig")

display(public_indicators[["indicator_id", "source_name", "risk_theme", "risk_label", "recommended_feature"]])
display(external_refs[["reference_id", "source_name", "case_number", "risk_theme"]])

## 4. Bagging 모델 학습 방식

모델은 수치형, 범주형, 불리언, 텍스트 피처를 함께 사용합니다.

- 수치형: 전세가율, 부채비율, 근저당, 선순위채권, 시세 괴리율
- 범주형: 계약 유형, 주택 유형, 지역, 법률 카테고리
- 불리언: 압류, 가압류, 신탁, 위반건축물, 보증보험 가능 여부 등
- 텍스트: 판례/특약/상황 설명에서 추출한 위험 키워드

평가는 같은 `source_case_number`에서 파생된 행이 train과 holdout에 동시에 들어가지 않도록 그룹 분리했습니다.

In [ ]:
metadata = json.loads((MODELS_DIR / "metadata.json").read_text(encoding="utf-8"))
metrics = json.loads((LEARNING_DIR / "metrics.json").read_text(encoding="utf-8"))

summary = {
    "algorithm": metadata["algorithm"],
    "base_estimator": metadata["base_estimator"],
    "n_estimators": metadata["n_estimators"],
    "rows": metadata["rows"],
    "train_rows": metadata["train_rows"],
    "validation_rows": metadata["validation_rows"],
    "holdout_rows": metadata["holdout_rows"],
    "validation_macro_f1": metrics["validation"]["macro_f1"],
    "holdout_macro_f1": metrics["holdout"]["macro_f1"],
}
summary

In [ ]:
cm = pd.read_csv(LEARNING_DIR / "confusion_matrix.csv", encoding="utf-8-sig", index_col=0)
classification_report = (LEARNING_DIR / "classification_report.txt").read_text(encoding="utf-8")

display(cm)
print(classification_report)

## 5. 수동 시나리오 검증

실제 사용자가 넣을 법한 계약 정보를 22개 만들어 데이터셋에 없는 입력에서도 점수 범위가 맞는지 확인했습니다. 특히 낮은 전세가율+압류, 낮은 부채비율+신탁, 높은 전세가율 단독, 임대인 명의 불일치, 전입신고 지연/당일 근저당, 안전한 대리계약, 체크박스 누락 압류/신탁/보증보험 불가 구어체 입력 같은 반례와 현장패턴 입력을 포함합니다. 추가학습에서는 보호요건 완료 문장과 신탁·압류·체납·이중계약·보증보험 불가 같은 사용자 자연어 표현도 별도 피처로 반영했습니다. 이 파일은 테스트에서도 사용됩니다.

In [ ]:
manual_predictions = pd.read_csv(LEARNING_DIR / "manual_scenario_predictions.csv", encoding="utf-8-sig")
manual_predictions[[
    "scenario_id",
    "name",
    "risk_score",
    "risk_grade",
    "model_predicted_grade",
    "prob_safe",
    "prob_caution",
    "prob_danger",
]]

## 6. 단일 계약 예측 예시

아래 코드는 웹/API가 내부적으로 사용하는 `RiskScorer`를 직접 호출합니다. 최종 점수는 Bagging 모델 확률과 명시적 위험 규칙을 혼합합니다.

In [ ]:
from risk_detector.risk.scorer import RiskScorer

scorer = RiskScorer()

safe_payload = {
    "contract_type": "jeonse",
    "property_type": "apartment",
    "region": "수도권",
    "deposit_million": 240,
    "estimated_market_price_million": 560,
    "mortgage_million": 0,
    "senior_claim_million": 0,
    "guarantee_insurance_available": True,
    "fixed_date_ready": True,
    "move_in_ready": True,
    "broker_explained_rights": True,
    "nearby_market_gap_percent": -4,
    "special_clause_text": "잔금 전 등기부 재확인 및 권리침해 발생 시 해제 특약.",
    "user_situation_text": "실거래가와 비교해 보증금이 낮고 보증보험 가입 가능하다.",
}

danger_payload = {
    "contract_type": "jeonse",
    "property_type": "villa",
    "region": "수도권",
    "deposit_million": 270,
    "estimated_market_price_million": 300,
    "mortgage_million": 80,
    "senior_claim_million": 30,
    "provisional_seizure": True,
    "landlord_multiple_properties": True,
    "broker_advertising_issue": True,
    "suspicious_special_clause": True,
    "guarantee_insurance_available": False,
    "broker_explained_rights": False,
    "special_clause_text": "임차인은 임대인의 담보 제공 및 채권양도에 이의를 제기하지 않는다.",
    "user_situation_text": "사회초년생이 시세보다 높은 보증금의 신축 빌라 전세계약을 앞두고 있다.",
}

safe_result = scorer.score(safe_payload)
danger_result = scorer.score(danger_payload)

pd.DataFrame([
    {
        "case": "safe_example",
        "risk_score": safe_result["risk_score"],
        "risk_grade": safe_result["risk_grade"],
        "model_grade": safe_result["model_predicted_grade"],
        "probabilities": safe_result["model_probabilities"],
    },
    {
        "case": "danger_example",
        "risk_score": danger_result["risk_score"],
        "risk_grade": danger_result["risk_grade"],
        "model_grade": danger_result["model_predicted_grade"],
        "probabilities": danger_result["model_probabilities"],
    },
])

## 7. 웹 시연 화면

아래 이미지는 최종 로컬 웹 시연을 브라우저에서 검증한 결과입니다.

![desktop demo](web_demo_result_desktop_final.png)

웹 실행 명령:

```bash
PYTHONPATH=src .venv/bin/python scripts/run_web.py --host 127.0.0.1 --port 8765
```

## 8. 재학습 명령

데이터를 다시 생성하고 모델을 새로 학습하려면 프로젝트 루트에서 아래 명령을 실행합니다.

```bash
PYTHONPATH=src .venv/bin/python scripts/build_datasets.py
PYTHONPATH=src .venv/bin/python scripts/train_model.py
PYTHONPATH=src .venv/bin/python -m pytest
```

재학습 후 `models/metadata.json`, `학습과정/metrics.json`, `학습과정/training_report.md`가 갱신됩니다.